# QSAR Aromatase — 16 Models × 12 Fingerprints × 2 Splits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dom-castaneda/qsar-aromatase/blob/master/notebooks/colab_qsar_models.ipynb)

**Target**: pchembl_value regression for Aromatase (CYP19A1) inhibitors
**Total runs**: 16 × 12 × 2 = 384 model fits + 10-fold CV
**GPU**: cuML + XGBoost CUDA | **Fallback**: sklearn CPU
**Metrics**: R², RMSE, MAE on train, 10-fold CV, and test sets


## 1. Setup — Download Data & Install

In [ ]:
import os, subprocess, sys

# Download data from GitHub
if not os.path.exists("data"):
    !wget -q "https://github.com/dom-castaneda/qsar-aromatase/raw/master/data.zip" -O data.zip
    !unzip -qo data.zip
    print("Data extracted.")
else:
    print("Data already present.")

# Install XGBoost
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

# Try cuML (GPU)
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                          "--extra-index-url=https://pypi.nvidia.com", "cuml-cu12"])
    print("cuML installed")
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                              "--extra-index-url=https://pypi.nvidia.com", "cuml-cu11"])
        print("cuML (cu11) installed")
    except Exception:
        print("cuML not available - using sklearn")


## 2. Imports with GPU/CPU Fallback

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict

USE_GPU = False
try:
    import cuml
    from cuml.linear_model import Ridge as cuRidge, Lasso as cuLasso, ElasticNet as cuElasticNet
    from cuml.neighbors import KNeighborsRegressor as cuKNN
    from cuml.svm import SVR as cuSVR
    from cuml.ensemble import RandomForestRegressor as cuRF
    # Validate GPU
    import cupy as cp
    cp.zeros(10)
    _t = cuRidge(alpha=1.0)
    _t.fit(cp.zeros((10,3), dtype=cp.float32), cp.zeros(10, dtype=cp.float32))
    del _t
    USE_GPU = True
    print(f"GPU VALIDATED - cuML {cuml.__version__}")
except Exception as e:
    print(f"CPU mode: {e}")

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, AdaBoostRegressor,
                              HistGradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

XGB_DEVICE = "cuda" if USE_GPU else "cpu"
print(f"XGBoost: {XGB_DEVICE} | Mode: {'GPU' if USE_GPU else 'CPU'}")


## 3. Load Data

In [ ]:
RANDOM_STATE = 42
N_FOLDS = 10
BASE = "data"

FP_NAMES = {
    "MACCS": "fingerprints_maccs.csv",
    "PubChem": "fingerprints_pubchem.csv",
    "Substructure": "fingerprints_substruct.csv",
    "SubstructureCount": "fingerprints_substruct_count.csv",
    "KR": "fingerprints_kr.csv",
    "KR_Count": "fingerprints_kr_count.csv",
    "AtomPairs2D": "fingerprints_atompairs2d.csv",
    "AP2D_Count": "fingerprints_atompairs2d_count.csv",
    "CDK_FP": "fingerprints_cdk_fp.csv",
    "CDK_Extended": "fingerprints_cdk_extended.csv",
    "CDK_GraphOnly": "fingerprints_cdk_graphonly.csv",
    "EState": "fingerprints_estate.csv",
}

SPLITS = {
    "Random": (f"{BASE}/splits/random_train.csv", f"{BASE}/splits/random_test.csv"),
    "Kennard-Stone": (f"{BASE}/splits/kennard_stone_train.csv", f"{BASE}/splits/kennard_stone_test.csv"),
}

df_full = pd.read_csv(f"{BASE}/processed/aromatase_bioactivity_clean.csv")
mask = (df_full["standard_relation"] == "=") & df_full["pchembl_value"].notna()
df = df_full[mask].reset_index(drop=True)
print(f"Dataset: {len(df)} molecules")

fp_data = {}
for fp_name, fp_file in FP_NAMES.items():
    fp_full = pd.read_csv(f"{BASE}/fingerprints_filtered/{fp_file}")
    fp_filtered = fp_full[mask.values].reset_index(drop=True)
    fp_cols = [c for c in fp_filtered.columns if c != "molecule_chembl_id"]
    X = np.nan_to_num(fp_filtered[fp_cols].values.astype(np.float32), nan=0.0)
    fp_data[fp_name] = X
    print(f"  {fp_name:<20} {X.shape[1]:>5} features")

split_masks = {}
for split_name, (train_file, test_file) in SPLITS.items():
    train_ids = set(pd.read_csv(train_file)["molecule_chembl_id"])
    test_ids = set(pd.read_csv(test_file)["molecule_chembl_id"])
    split_masks[split_name] = (
        df["molecule_chembl_id"].isin(train_ids).values,
        df["molecule_chembl_id"].isin(test_ids).values,
    )

y_all = df["pchembl_value"].values
print(f"\nReady: {len(FP_NAMES)*len(SPLITS)*16} runs")


## 4. Model Builder

In [ ]:
def build_models():
    models = []
    if USE_GPU:
        models += [("Ridge", cuRidge(alpha=1.0)), ("Lasso", cuLasso(alpha=0.1)),
                   ("ElasticNet", cuElasticNet(alpha=0.1, l1_ratio=0.5))]
    else:
        models += [("Ridge", Ridge(alpha=1.0)), ("Lasso", Lasso(alpha=0.1, max_iter=5000)),
                   ("ElasticNet", ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000))]
    models += [("Bayesian Ridge", BayesianRidge()), ("PLS", PLSRegression(n_components=10))]
    models.append(("KNN", cuKNN(n_neighbors=5) if USE_GPU else KNeighborsRegressor(n_neighbors=5)))
    models.append(("SVR (RBF)", cuSVR(kernel="rbf", C=1.0, epsilon=0.1) if USE_GPU else SVR(kernel="rbf", C=1.0, epsilon=0.1)))
    models += [("Kernel Ridge (RBF)", KernelRidge(alpha=1.0, kernel="rbf")),
               ("Decision Tree", DecisionTreeRegressor(random_state=RANDOM_STATE))]
    models.append(("Random Forest", cuRF(n_estimators=500, random_state=RANDOM_STATE) if USE_GPU
                   else RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)))
    models += [("Extra Trees", ExtraTreesRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)),
               ("Gradient Boosting", GradientBoostingRegressor(n_estimators=500, random_state=RANDOM_STATE)),
               ("XGBoost", XGBRegressor(n_estimators=500, learning_rate=0.1, random_state=RANDOM_STATE, verbosity=0, device=XGB_DEVICE, n_jobs=-1)),
               ("Hist Gradient Boosting", HistGradientBoostingRegressor(max_iter=500, random_state=RANDOM_STATE)),
               ("AdaBoost", AdaBoostRegressor(n_estimators=500, random_state=RANDOM_STATE)),
               ("MLP", MLPRegressor(hidden_layer_sizes=(256,128), max_iter=500, random_state=RANDOM_STATE, early_stopping=True))]
    return models

def compute_metrics(y_true, y_pred):
    return r2_score(y_true, y_pred), np.sqrt(mean_squared_error(y_true, y_pred)), mean_absolute_error(y_true, y_pred)

print(f"{len(build_models())} models. GPU={USE_GPU}")


## 5. Training Loop

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
all_results = []
total_runs = len(SPLITS) * len(FP_NAMES) * 16
run_count = 0
t_start = time.time()

for split_name, (train_mask, test_mask) in split_masks.items():
    y_train, y_test = y_all[train_mask], y_all[test_mask]
    for fp_name, X_fp in fp_data.items():
        X_train, X_test = X_fp[train_mask], X_fp[test_mask]
        for model_name, model in build_models():
            run_count += 1
            t0 = time.time()
            try:
                try:
                    y_cv = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=1)
                    r2_cv, rmse_cv, mae_cv = compute_metrics(y_train, y_cv)
                except: r2_cv, rmse_cv, mae_cv = np.nan, np.nan, np.nan
                model.fit(X_train, y_train)
                yp_tr = np.asarray(model.predict(X_train)).ravel()
                yp_te = np.asarray(model.predict(X_test)).ravel()
                r2_tr, rmse_tr, mae_tr = compute_metrics(y_train, yp_tr)
                r2_te, rmse_te, mae_te = compute_metrics(y_test, yp_te)
                all_results.append({"Split": split_name, "Fingerprint": fp_name, "Model": model_name,
                    "R2_train": r2_tr, "RMSE_train": rmse_tr, "MAE_train": mae_tr,
                    "R2_CV": r2_cv, "RMSE_CV": rmse_cv, "MAE_CV": mae_cv,
                    "R2_test": r2_te, "RMSE_test": rmse_te, "MAE_test": mae_te,
                    "Time_s": time.time()-t0})
            except Exception as e:
                all_results.append({"Split": split_name, "Fingerprint": fp_name, "Model": model_name,
                    "R2_train": np.nan, "RMSE_train": np.nan, "MAE_train": np.nan,
                    "R2_CV": np.nan, "RMSE_CV": np.nan, "MAE_CV": np.nan,
                    "R2_test": np.nan, "RMSE_test": np.nan, "MAE_test": np.nan,
                    "Time_s": time.time()-t0})
                print(f"  FAILED: {split_name}/{fp_name}/{model_name}: {e}")
            if run_count % 48 == 0 or run_count == total_runs:
                et = time.time()-t_start
                print(f"  [{run_count}/{total_runs}] {split_name}|{fp_name}|{model_name} "
                      f"R2={all_results[-1]['R2_test']:.4f} ({et:.0f}s, ETA ~{(et/run_count)*(total_runs-run_count)/60:.0f}min)")

print(f"\nDONE: {run_count} runs in {(time.time()-t_start)/60:.1f} min")


## 6. Results

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv("results_all_models.csv", index=False)
n_ok = results_df["R2_test"].notna().sum()
print(f"Saved: results_all_models.csv ({n_ok}/{len(results_df)} successful)")
results_df.sort_values("R2_test", ascending=False).head(20)


## 7. Best Models & Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

valid = results_df.dropna(subset=["R2_test"])
best = valid.loc[valid["R2_test"].idxmax()]
print(f"OVERALL BEST: {best['Model']} on {best['Fingerprint']} ({best['Split']})")
print(f"  R2={best['R2_test']:.4f}, RMSE={best['RMSE_test']:.4f}, MAE={best['MAE_test']:.4f}")

for split_name in SPLITS:
    sub = valid[valid["Split"] == split_name]
    pivot = sub.pivot_table(index="Model", columns="Fingerprint", values="R2_test")
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=0.7,
                linewidths=0.5, ax=ax)
    ax.set_title(f"Test R2 - {split_name} Split")
    plt.tight_layout()
    plt.show()


## 8. Download

In [ ]:
from google.colab import files
files.download("results_all_models.csv")
